### 01 — Import Python libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

print("Python tools are ready!")

### 02 — Open the patient CSV file

In [ ]:
df_raw = pd.read_csv("/content/NOVIRA26.csv")

display(df_raw.head())

### 03 — Explore the patient records

In [ ]:
print("Number of records:", len(df_raw))
print("Number of unique patients:", df_raw["patient_id"].nunique())

display(df_raw[["age", "SpO2_pct", "CRP_mg_L"]].describe().round(1))

### 04 — Check data quality

In [ ]:
print("Duplicate patient records:", df_raw.duplicated(subset="patient_id").sum())
print("Missing CRP values:", df_raw["CRP_mg_L"].isna().sum())

print("Valid oxygen readings (True / False):")
display(df_raw["SpO2_pct"].between(60, 100).value_counts())

### 05 — Clean the patient records

In [ ]:
df = df_raw.drop_duplicates(subset="patient_id").copy()
df["SpO2_pct"] = df["SpO2_pct"].where(df["SpO2_pct"].between(60, 100))

print("Patients after removing duplicates:", len(df))
print("Oxygen values needing review:", df["SpO2_pct"].isna().sum())

### 06 — Plot daily patient records

In [ ]:
df["sample_date"] = pd.to_datetime(df["sample_date"])
daily_records = df.groupby("sample_date").size()

plt.figure(figsize=(10, 4))
plt.plot(daily_records.index, daily_records.values, color="#287C99")
plt.title("NOVIRA-26: Daily patient records")
plt.xlabel("Sample date")
plt.ylabel("Number of records")
plt.show()

### 07 — Visualize CRP levels

In [ ]:
plt.figure(figsize=(9, 4))
sns.violinplot(data=df, x="baseline_risk", y="CRP_mg_L",
               order=["Low", "Moderate", "High"], color="#69AFA8")
plt.title("CRP levels by baseline risk")
plt.xlabel("Baseline risk")
plt.ylabel("CRP (mg/L)")
plt.show()

### 08 — Compare treatments

In [ ]:
df["hospitalized_percent"] = df["hospitalized"] * 100
overall_rates = df.groupby("treatment")["hospitalized_percent"].mean()

display(overall_rates.round(1))

overall_rates.plot(kind="bar", color=["#D96847", "#447DB1"])
plt.title("Overall hospitalization rate by treatment")
plt.xlabel("Treatment")
plt.ylabel("Hospitalization rate (%)")
plt.xticks(rotation=0)
plt.show()

### 09 — Compare baseline risk between treatments

In [ ]:
df["high_risk_percent"] = (df["baseline_risk"] == "High") * 100
high_risk_rates = df.groupby("treatment")["high_risk_percent"].mean()

display(high_risk_rates.round(1))

high_risk_rates.plot(kind="bar", color=["#D96847", "#447DB1"])
plt.title("High-risk patients in each treatment group")
plt.xlabel("Treatment")
plt.ylabel("High-risk patients (%)")
plt.xticks(rotation=0)
plt.show()

### 10 — Compare treatments within each risk group

In [ ]:
rates_by_risk = df.groupby(["baseline_risk", "treatment"])["hospitalized_percent"].mean()

display(rates_by_risk.round(1))

plt.figure(figsize=(9, 4))
sns.barplot(data=df, x="baseline_risk", y="hospitalized_percent",
            hue="treatment", order=["Low", "Moderate", "High"], errorbar=None)
plt.title("Hospitalization rates within each risk group")
plt.xlabel("Baseline risk")
plt.ylabel("Hospitalization rate (%)")
plt.show()

### 11 — Save the results as CSV files

In [ ]:
df.to_csv("NOVIRA26_cleaned.csv", index=False)
rates_by_risk.to_csv("NOVIRA26_rates_by_risk.csv")

print("CSV files saved. Download them from the Colab Files panel.")

### 12 — Final hospital comparison

In [ ]:
hospital_rates = df.groupby("hospital")["hospitalized_percent"].mean()

print("Patients analyzed:", len(df))
display(hospital_rates.round(1))

hospital_rates.plot(kind="bar", color="#287C99")
plt.title("NOVIRA-26: Hospital comparison")
plt.xlabel("Hospital")
plt.ylabel("Hospitalization rate (%)")
plt.xticks(rotation=20)
plt.show()

print("Remember: Observational data cannot prove that a treatment works.")